### Vecort stors and retrievers
This video tutorial will familiarize you with Langchain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data--> from(vector) databases and other sources-->for integration with LLM workplows They are important for application that fetch data to be respond over as part of model inference, as in the case of retrieval-augmented genration.\

We will cover
* Documents
* Vector stores
* Retrievers

## Documents 
Langchain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has teo attributs:

- page_content: a string representing the content.
- metadata: a dict containing arbitrary metadata. The metadat attribute can capture information about the source of the document, its relationship to other documents, and other infomation. Note that an individual Document object often represents a chunk of a large document.


In [1]:
from langchain_core.documents import Document

Documents = [
    Document(
        page_content="Doges are great companions, know for their loyalty and friendliness.",
        metadata={"source":"mammal-pets-doc"}
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source":"mammal-pets-doc"}
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source":"fish-pets-doc"}
    ),
    Document(
        page_content="Parrots are intelligents birds capable of mimicking human speech.",
        metadata={"source":"bird-pets-doc"}
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source":"mammal-pets-doc"}
    ),

]

In [2]:
Documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Doges are great companions, know for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligents birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm=ChatGroq(groq_proxy=groq_api_key,model="llama-3.1-8b-instant")


c:\Users\himan\anaconda3\envs\llmenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1083.17it/s]


In [5]:
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(Documents,embedding=embeddings)
vectorstore

In [6]:
vectorstore.similarity_search("cat")

[Document(id='6e9c3ffb-6646-437e-a355-353b666e2d93', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='2cf2281a-0896-42f7-91bf-02ad1f926fd4', metadata={'source': 'mammal-pets-doc'}, page_content='Doges are great companions, know for their loyalty and friendliness.'),
 Document(id='b14e4d3e-6de1-41a1-bb0d-8b93b704fb64', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='4d03cb6f-076c-47c7-95a5-f3db4ee21a95', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligents birds capable of mimicking human speech.')]

In [7]:
## Async Query
await vectorstore.asimilarity_search("cat")

[Document(id='6e9c3ffb-6646-437e-a355-353b666e2d93', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='2cf2281a-0896-42f7-91bf-02ad1f926fd4', metadata={'source': 'mammal-pets-doc'}, page_content='Doges are great companions, know for their loyalty and friendliness.'),
 Document(id='b14e4d3e-6de1-41a1-bb0d-8b93b704fb64', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='4d03cb6f-076c-47c7-95a5-f3db4ee21a95', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligents birds capable of mimicking human speech.')]

In [8]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='6e9c3ffb-6646-437e-a355-353b666e2d93', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351058006286621),
 (Document(id='2cf2281a-0896-42f7-91bf-02ad1f926fd4', metadata={'source': 'mammal-pets-doc'}, page_content='Doges are great companions, know for their loyalty and friendliness.'),
  1.5151926279067993),
 (Document(id='b14e4d3e-6de1-41a1-bb0d-8b93b704fb64', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956902503967285),
 (Document(id='4d03cb6f-076c-47c7-95a5-f3db4ee21a95', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligents birds capable of mimicking human speech.'),
  1.6571002006530762)]

### Retrievers
Langchain Vectorstore object do not subclass Runnable, and also so cannot immeditely be integrated into Langchain Expression Language chains.


LangChain retrievers are Runnable, So they implement a standard set of methods(e.g: synchronous and asynchronous invoke and batch operation) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retrivers. If we chooose whate methode we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [9]:
from typing import List 
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='6e9c3ffb-6646-437e-a355-353b666e2d93', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='2cf2281a-0896-42f7-91bf-02ad1f926fd4', metadata={'source': 'mammal-pets-doc'}, page_content='Doges are great companions, know for their loyalty and friendliness.')]]

VectorStore implement an as_retriver methods that will genrate a Retriver, specifically a VectorStoreRetriever. These retriever include specific search_kwargs attributs that identify what methods of the sunderlying vectors store to call, and how to parameterize them. Fro instance, we can replicate the above with the following

In [12]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])

[[Document(id='6e9c3ffb-6646-437e-a355-353b666e2d93', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='2cf2281a-0896-42f7-91bf-02ad1f926fd4', metadata={'source': 'mammal-pets-doc'}, page_content='Doges are great companions, know for their loyalty and friendliness.')]]

In [ ]:
### RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message="""
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt=ChatPromptTemplate.from_messages(["human",message])

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm

response=rag_chain.invoke("tell me about dogs")
print(response)

content='Dogs are great companions, known for their loyalty and friendliness.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 115, 'total_tokens': 130, 'completion_time': 0.012194547, 'completion_tokens_details': None, 'prompt_time': 0.006768154, 'prompt_tokens_details': None, 'queue_time': 0.052283835, 'total_time': 0.018962701}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019df933-3abb-71d0-bd26-57ef82a6bc07-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 115, 'output_tokens': 15, 'total_tokens': 130}
